In [ ]:
#!/usr/bin/env python3
"""
plot.py — Offline graph generator for perf_metrics.csv

Reads the CSV produced by collector.c and generates a set of matplotlib
figures. Each figure is saved as a PNG. No live view, no web server.

Usage:
    python3 plot.py perf_metrics.csv
    python3 plot.py perf_metrics.csv --pid 1234         # filter one PID
    python3 plot.py perf_metrics.csv --out ./plots/     # output directory
    python3 plot.py perf_metrics.csv --show             # pop up windows too
    python3 plot.py perf_metrics.csv --label cpu_bound  # filter one label
"""

import sys
import argparse
import os
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import matplotlib
matplotlib.use("Agg")           # no display needed
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.ticker as ticker
from matplotlib.patches import Patch
import numpy as np

# ─── colour scheme (dark terminal aesthetic) ──────────────────────────────────
BG    = "#0d1117"
AXES  = "#161b22"
GRID  = "#21262d"
TEXT  = "#c9d1d9"
MUTED = "#484f58"

LABEL_COLORS = {
    "cpu_bound":    "#f87171",
    "memory_bound": "#60a5fa",
    "io_bound":     "#4ade80",
    "contention":   "#fb923c",
    "mixed":        "#c084fc",
    "unknown":      "#6b7280",
}
LINE_PALETTE = ["#60a5fa", "#f87171", "#4ade80", "#fb923c", "#c084fc",
                "#facc15", "#34d399", "#f472b6"]

# ─── matplotlib style setup ───────────────────────────────────────────────────

def apply_style():
    plt.rcParams.update({
        "figure.facecolor":     BG,
        "axes.facecolor":       AXES,
        "axes.edgecolor":       GRID,
        "axes.labelcolor":      TEXT,
        "axes.titlecolor":      TEXT,
        "xtick.color":          MUTED,
        "ytick.color":          MUTED,
        "text.color":           TEXT,
        "grid.color":           GRID,
        "grid.linewidth":       0.6,
        "lines.linewidth":      1.4,
        "font.family":          "monospace",
        "font.size":            10,
        "axes.titlesize":       11,
        "axes.labelsize":       9,
        "legend.facecolor":     AXES,
        "legend.edgecolor":     GRID,
        "legend.labelcolor":    TEXT,
        "legend.fontsize":      8,
        "figure.dpi":           120,
        "savefig.dpi":          150,
        "savefig.bbox":         "tight",
        "savefig.facecolor":    BG,
    })

# ─── data loading ─────────────────────────────────────────────────────────────

def load(csv_path, pid_filter=None, label_filter=None):
    print(f"[plot] loading {csv_path} ...")
    df = pd.read_csv(csv_path)
    print(f"[plot] {len(df)} rows, {len(df.columns)} columns")

    # convert ns columns to µs where useful
    ns_cols = [c for c in df.columns if c.endswith("_ns")]
    for c in ns_cols:
        us_col = c.replace("_ns", "_us")
        df[us_col] = df[c] / 1000.0

    # derived columns
    if "hw_instructions" in df and "hw_cycles" in df:
        df["ipc_derived"] = df["hw_instructions"] / df["hw_cycles"].replace(0, float("nan"))
    if "hw_llc_misses" in df and "hw_llc_refs" in df:
        df["llc_miss_rate"] = df["hw_llc_misses"] / df["hw_llc_refs"].replace(0, float("nan"))

    # relative time axis
    if "timestamp_s" in df:
        df["t_rel"] = df["timestamp_s"] - df["timestamp_s"].min()

    # total page faults
    df["total_faults"] = df.get("minor_faults", 0) + df.get("kernel_faults", 0)

    # total lock contentions
    df["total_lock_contentions"] = (
        df.get("mutex_contentions", 0) +
        df.get("rwsem_read_contentions", 0) +
        df.get("rwsem_write_contentions", 0)
    )

    if pid_filter is not None:
        df = df[df["pid"] == int(pid_filter)]
        print(f"[plot] filtered to pid={pid_filter}: {len(df)} rows")

    if label_filter:
        df = df[df["label"] == label_filter]
        print(f"[plot] filtered to label={label_filter}: {len(df)} rows")

    if df.empty:
        print("[plot] ERROR: no rows after filtering")
        sys.exit(1)

    return df


# ─── helper: add label background bands ───────────────────────────────────────

def add_label_bands(ax, df):
    """Shade time axis by workload label."""
    if "label" not in df.columns or "t_rel" not in df.columns:
        return
    prev_label = None
    start = None
    for _, row in df.drop_duplicates("t_rel").sort_values("t_rel").iterrows():
        lbl = row["label"]
        t   = row["t_rel"]
        if lbl != prev_label:
            if prev_label is not None:
                c = LABEL_COLORS.get(prev_label, MUTED)
                ax.axvspan(start, t, alpha=0.08, color=c, linewidth=0)
            start = t
            prev_label = lbl
    if prev_label and start is not None:
        t_end = df["t_rel"].max()
        c = LABEL_COLORS.get(prev_label, MUTED)
        ax.axvspan(start, t_end, alpha=0.08, color=c, linewidth=0)


def legend_patches(df):
    labels = df["label"].unique() if "label" in df.columns else []
    return [Patch(color=LABEL_COLORS.get(l, MUTED), alpha=0.6, label=l) for l in labels]


# ─── per-PID aggregation ──────────────────────────────────────────────────────

def agg_by_time(df, cols, groupby="t_rel"):
    """Sum numeric columns per time bucket, keeping label."""
    numeric = [c for c in cols if c in df.columns]
    grp = df.groupby(groupby)
    result = grp[numeric].sum()
    # attach most common label per bucket
    if "label" in df.columns:
        result["label"] = grp["label"].agg(lambda x: x.mode()[0])
    return result.reset_index()


# ─── FIGURE 1: CPU scheduling ─────────────────────────────────────────────────

def fig_scheduling(df, out_dir):
    print("[plot] figure 1: scheduling")
    apply_style()

    agg = agg_by_time(df, [
        "ctx_switches", "voluntary_switches", "involuntary_switches",
        "cpu_migrations", "avg_runq_latency_us", "max_runq_latency_us",
        "total_runtime_ns",
    ])

    fig, axes = plt.subplots(3, 1, figsize=(14, 9), sharex=True)
    fig.suptitle("CPU Scheduling Metrics", fontsize=13, fontweight="normal", y=0.98)

    t = agg["t_rel"]

    # Panel 1: context switches breakdown
    ax = axes[0]
    if "involuntary_switches" in agg and "voluntary_switches" in agg:
        ax.fill_between(t, agg["involuntary_switches"], alpha=0.6,
                        color=LINE_PALETTE[0], label="involuntary (CPU pressure)")
        ax.fill_between(t, agg["voluntary_switches"], alpha=0.4,
                        color=LINE_PALETTE[1], label="voluntary (I/O / lock wait)")
        ax.plot(t, agg["ctx_switches"], color=TEXT, linewidth=0.8, label="total")
    ax.set_ylabel("ctx switches / interval")
    ax.legend(loc="upper right")
    ax.grid(True, axis="y")
    add_label_bands(ax, agg)

    # Panel 2: run queue latency
    ax = axes[1]
    if "avg_runq_latency_us" in agg:
        ax.plot(t, agg["avg_runq_latency_us"], color=LINE_PALETTE[2], label="avg runq latency µs")
    if "max_runq_latency_us" in agg:
        ax.plot(t, agg["max_runq_latency_us"], color=LINE_PALETTE[1],
                linestyle="--", linewidth=0.8, label="max runq latency µs")
    ax.set_ylabel("µs")
    ax.legend(loc="upper right")
    ax.grid(True, axis="y")
    add_label_bands(ax, agg)

    # Panel 3: migrations
    ax = axes[2]
    if "cpu_migrations" in agg:
        ax.bar(t, agg["cpu_migrations"], width=t.diff().median(),
               color=LINE_PALETTE[4], alpha=0.7, label="CPU migrations")
    ax.set_ylabel("migrations / interval")
    ax.set_xlabel("time (s)")
    ax.legend(loc="upper right")
    ax.grid(True, axis="y")
    add_label_bands(ax, agg)

    # add label legend
    patches = legend_patches(df)
    if patches:
        fig.legend(handles=patches, loc="lower center", ncol=len(patches),
                   frameon=False, fontsize=8, bbox_to_anchor=(0.5, 0.01))

    plt.tight_layout(rect=[0, 0.04, 1, 0.97])
    path = os.path.join(out_dir, "01_scheduling.png")
    plt.savefig(path); plt.close()
    print(f"[plot] saved {path}")


# ─── FIGURE 2: Memory ─────────────────────────────────────────────────────────

def fig_memory(df, out_dir):
    print("[plot] figure 2: memory")
    apply_style()

    agg = agg_by_time(df, [
        "minor_faults", "kernel_faults", "total_faults",
        "kmalloc_count", "kfree_count",
        "total_alloc_bytes", "total_free_bytes",
        "large_page_allocs",
    ])

    fig, axes = plt.subplots(3, 1, figsize=(14, 9), sharex=True)
    fig.suptitle("Memory Metrics", fontsize=13, fontweight="normal", y=0.98)
    t = agg["t_rel"]

    # Panel 1: page faults
    ax = axes[0]
    ax.fill_between(t, agg.get("minor_faults", 0), alpha=0.5,
                    color=LINE_PALETTE[0], label="minor faults")
    ax.fill_between(t, agg.get("kernel_faults", 0), alpha=0.6,
                    color=LINE_PALETTE[1], label="kernel faults")
    ax.set_ylabel("faults / interval")
    ax.legend(loc="upper right"); ax.grid(True, axis="y")
    add_label_bands(ax, agg)

    # Panel 2: kmalloc / kfree rate
    ax = axes[1]
    ax.plot(t, agg.get("kmalloc_count", 0), color=LINE_PALETTE[2], label="kmalloc count")
    ax.plot(t, agg.get("kfree_count", 0), color=LINE_PALETTE[3],
            linestyle="--", label="kfree count")
    ax.set_ylabel("calls / interval"); ax.legend(loc="upper right"); ax.grid(True, axis="y")
    add_label_bands(ax, agg)

    # Panel 3: alloc bytes + large page pressure
    ax = axes[2]
    ax2 = ax.twinx()
    ax.fill_between(t, agg.get("total_alloc_bytes", 0) / 1024,
                    color=LINE_PALETTE[0], alpha=0.4, label="alloc KB")
    ax2.plot(t, agg.get("large_page_allocs", 0), color=LINE_PALETTE[1],
             linewidth=1, label="large page allocs (pressure)")
    ax.set_ylabel("KB / interval"); ax2.set_ylabel("large allocs", color=LINE_PALETTE[1])
    ax.set_xlabel("time (s)")
    ax.legend(loc="upper left"); ax2.legend(loc="upper right")
    ax.grid(True, axis="y")
    add_label_bands(ax, agg)

    patches = legend_patches(df)
    if patches:
        fig.legend(handles=patches, loc="lower center", ncol=len(patches),
                   frameon=False, fontsize=8, bbox_to_anchor=(0.5, 0.01))

    plt.tight_layout(rect=[0, 0.04, 1, 0.97])
    path = os.path.join(out_dir, "02_memory.png")
    plt.savefig(path); plt.close()
    print(f"[plot] saved {path}")


# ─── FIGURE 3: Syscall latency ────────────────────────────────────────────────

def fig_syscalls(df, out_dir):
    print("[plot] figure 3: syscalls")
    apply_style()

    agg = agg_by_time(df, [
        "syscall_count", "avg_syscall_latency_us", "max_syscall_latency_us",
        "read_count", "write_count", "read_bytes", "write_bytes",
        "futex_count", "avg_futex_latency_us",
        "epoll_count", "avg_epoll_latency_us",
        "poll_count", "syscall_error_count",
    ])

    fig, axes = plt.subplots(3, 1, figsize=(14, 9), sharex=True)
    fig.suptitle("Syscall & I/O Metrics", fontsize=13, fontweight="normal", y=0.98)
    t = agg["t_rel"]

    # Panel 1: average syscall latency (all + futex + epoll)
    ax = axes[0]
    for col, label, color in [
        ("avg_syscall_latency_us", "avg all syscalls µs", LINE_PALETTE[0]),
        ("avg_futex_latency_us",   "avg futex µs",        LINE_PALETTE[1]),
        ("avg_epoll_latency_us",   "avg epoll µs",        LINE_PALETTE[2]),
    ]:
        if col in agg.columns:
            ax.plot(t, agg[col], color=color, label=label)
    ax.set_ylabel("latency µs"); ax.legend(loc="upper right"); ax.grid(True, axis="y")
    add_label_bands(ax, agg)

    # Panel 2: read / write throughput
    ax = axes[1]
    ax2 = ax.twinx()
    ax.fill_between(t, agg.get("read_bytes", 0) / 1024,
                    color=LINE_PALETTE[0], alpha=0.4, label="read KB")
    ax.fill_between(t, agg.get("write_bytes", 0) / 1024,
                    color=LINE_PALETTE[2], alpha=0.4, label="write KB")
    ax2.plot(t, agg.get("syscall_error_count", 0),
             color=LINE_PALETTE[1], linewidth=0.8, linestyle=":", label="errors")
    ax.set_ylabel("KB / interval"); ax2.set_ylabel("errors", color=LINE_PALETTE[1])
    ax.legend(loc="upper left"); ax2.legend(loc="upper right")
    ax.grid(True, axis="y")
    add_label_bands(ax, agg)

    # Panel 3: syscall type distribution (stacked)
    ax = axes[2]
    bottom = np.zeros(len(t))
    for col, color, label in [
        ("read_count",  LINE_PALETTE[0], "read"),
        ("write_count", LINE_PALETTE[2], "write"),
        ("futex_count", LINE_PALETTE[1], "futex"),
        ("epoll_count", LINE_PALETTE[4], "epoll"),
        ("poll_count",  LINE_PALETTE[5], "poll"),
    ]:
        vals = agg.get(col, 0).values
        ax.fill_between(t, bottom, bottom + vals, alpha=0.6, color=color, label=label)
        bottom = bottom + vals
    ax.set_ylabel("count / interval"); ax.set_xlabel("time (s)")
    ax.legend(loc="upper right"); ax.grid(True, axis="y")
    add_label_bands(ax, agg)

    patches = legend_patches(df)
    if patches:
        fig.legend(handles=patches, loc="lower center", ncol=len(patches),
                   frameon=False, fontsize=8, bbox_to_anchor=(0.5, 0.01))

    plt.tight_layout(rect=[0, 0.04, 1, 0.97])
    path = os.path.join(out_dir, "03_syscalls.png")
    plt.savefig(path); plt.close()
    print(f"[plot] saved {path}")


# ─── FIGURE 4: Lock contention ────────────────────────────────────────────────

def fig_locks(df, out_dir):
    print("[plot] figure 4: lock contention")
    apply_style()

    agg = agg_by_time(df, [
        "mutex_contentions", "avg_mutex_wait_us", "max_mutex_wait_us",
        "rwsem_read_contentions", "avg_rwsem_read_wait_us",
        "rwsem_write_contentions", "avg_rwsem_write_wait_us", "max_rwsem_write_wait_us",
        "total_lock_contentions",
    ])

    fig, axes = plt.subplots(2, 1, figsize=(14, 7), sharex=True)
    fig.suptitle("Lock Contention", fontsize=13, fontweight="normal", y=0.98)
    t = agg["t_rel"]

    ax = axes[0]
    for col, color, label in [
        ("mutex_contentions",        LINE_PALETTE[0], "mutex"),
        ("rwsem_read_contentions",   LINE_PALETTE[2], "rwsem read"),
        ("rwsem_write_contentions",  LINE_PALETTE[1], "rwsem write"),
    ]:
        if col in agg.columns:
            ax.plot(t, agg[col], color=color, label=label)
    ax.set_ylabel("contention events / interval")
    ax.legend(loc="upper right"); ax.grid(True, axis="y")
    add_label_bands(ax, agg)

    ax = axes[1]
    for col, color, label in [
        ("avg_mutex_wait_us",       LINE_PALETTE[0], "avg mutex wait µs"),
        ("max_mutex_wait_us",       LINE_PALETTE[1], "max mutex wait µs"),
        ("avg_rwsem_write_wait_us", LINE_PALETTE[2], "avg rwsem write wait µs"),
    ]:
        if col in agg.columns:
            ax.plot(t, agg[col], color=color,
                    linestyle="--" if "max" in col else "-", label=label)
    ax.set_ylabel("µs"); ax.set_xlabel("time (s)")
    ax.legend(loc="upper right"); ax.grid(True, axis="y")
    add_label_bands(ax, agg)

    patches = legend_patches(df)
    if patches:
        fig.legend(handles=patches, loc="lower center", ncol=len(patches),
                   frameon=False, fontsize=8, bbox_to_anchor=(0.5, 0.01))

    plt.tight_layout(rect=[0, 0.04, 1, 0.97])
    path = os.path.join(out_dir, "04_locks.png")
    plt.savefig(path); plt.close()
    print(f"[plot] saved {path}")


# ─── FIGURE 5: Hardware perf counters ────────────────────────────────────────

def fig_hw(df, out_dir):
    hw_cols = ["hw_cycles", "hw_instructions", "hw_llc_misses", "hw_llc_refs"]
    present = [c for c in hw_cols if c in df.columns and df[c].sum() > 0]
    if not present:
        print("[plot] no hw perf counter data — skipping figure 5")
        return

    print("[plot] figure 5: hardware counters")
    apply_style()

    agg = agg_by_time(df, hw_cols + ["ipc_derived", "llc_miss_rate"])
    fig, axes = plt.subplots(3, 1, figsize=(14, 9), sharex=True)
    fig.suptitle("Hardware Performance Counters", fontsize=13, fontweight="normal", y=0.98)
    t = agg["t_rel"]

    ax = axes[0]
    ax.plot(t, agg.get("hw_cycles", 0) / 1e9, color=LINE_PALETTE[0], label="cycles (G)")
    ax.plot(t, agg.get("hw_instructions", 0) / 1e9, color=LINE_PALETTE[2], label="instructions (G)")
    ax.set_ylabel("× 10⁹ / interval"); ax.legend(loc="upper right"); ax.grid(True, axis="y")
    add_label_bands(ax, agg)

    ax = axes[1]
    ax.plot(t, agg.get("ipc_derived", 0), color=LINE_PALETTE[3], label="IPC (instructions / cycle)")
    ax.axhline(y=1.0, color=MUTED, linewidth=0.6, linestyle=":")
    ax.set_ylabel("IPC"); ax.legend(loc="upper right"); ax.grid(True, axis="y")
    add_label_bands(ax, agg)

    ax = axes[2]
    ax.fill_between(t, agg.get("hw_llc_misses", 0) / 1e6, alpha=0.6,
                    color=LINE_PALETTE[1], label="LLC misses (M)")
    ax2 = ax.twinx()
    if "llc_miss_rate" in agg.columns:
        ax2.plot(t, agg["llc_miss_rate"] * 100, color=LINE_PALETTE[4],
                 linewidth=0.8, label="miss rate %")
        ax2.set_ylabel("miss rate %", color=LINE_PALETTE[4])
        ax2.legend(loc="upper right")
    ax.set_ylabel("LLC misses (M)"); ax.set_xlabel("time (s)")
    ax.legend(loc="upper left"); ax.grid(True, axis="y")
    add_label_bands(ax, agg)

    patches = legend_patches(df)
    if patches:
        fig.legend(handles=patches, loc="lower center", ncol=len(patches),
                   frameon=False, fontsize=8, bbox_to_anchor=(0.5, 0.01))

    plt.tight_layout(rect=[0, 0.04, 1, 0.97])
    path = os.path.join(out_dir, "05_hw_counters.png")
    plt.savefig(path); plt.close()
    print(f"[plot] saved {path}")


# ─── FIGURE 6: Bottleneck overview / correlation heatmap ─────────────────────

def fig_overview(df, out_dir):
    print("[plot] figure 6: overview / correlation")
    apply_style()

    feature_cols = [
        "ctx_switches", "involuntary_switches", "avg_runq_latency_us",
        "cpu_migrations", "minor_faults", "kernel_faults",
        "kmalloc_count", "avg_syscall_latency_us", "futex_count",
        "mutex_contentions", "total_lock_contentions",
        "hw_llc_misses", "ipc_derived",
    ]
    present = [c for c in feature_cols if c in df.columns]
    if len(present) < 3:
        print("[plot] not enough columns for overview — skipping figure 6")
        return

    fig = plt.figure(figsize=(16, 10))
    gs = gridspec.GridSpec(2, 2, figure=fig, hspace=0.4, wspace=0.3)
    fig.suptitle("Bottleneck Overview", fontsize=13, fontweight="normal")

    # ── subplot A: label distribution ──
    ax_a = fig.add_subplot(gs[0, 0])
    if "label" in df.columns:
        counts = df["label"].value_counts()
        colors = [LABEL_COLORS.get(l, MUTED) for l in counts.index]
        ax_a.barh(counts.index, counts.values, color=colors, alpha=0.8)
        ax_a.set_xlabel("row count")
        ax_a.set_title("label distribution")
        ax_a.grid(True, axis="x")

    # ── subplot B: per-label boxplot for ctx_switches ──
    ax_b = fig.add_subplot(gs[0, 1])
    if "label" in df.columns and "ctx_switches" in df.columns:
        labels_sorted = df["label"].unique()
        data = [df[df["label"] == l]["ctx_switches"].dropna().values for l in labels_sorted]
        bp = ax_b.boxplot(data, labels=labels_sorted, patch_artist=True,
                          medianprops={"color": TEXT, "linewidth": 1.5},
                          boxprops={"linewidth": 0.8},
                          whiskerprops={"linewidth": 0.8},
                          capprops={"linewidth": 0.8},
                          flierprops={"marker": ".", "markersize": 3})
        for patch, lbl in zip(bp["boxes"], labels_sorted):
            patch.set_facecolor(LABEL_COLORS.get(lbl, MUTED))
            patch.set_alpha(0.6)
        ax_b.set_ylabel("ctx_switches"); ax_b.set_title("ctx switches by label")
        ax_b.grid(True, axis="y")

    # ── subplot C: correlation heatmap ──
    ax_c = fig.add_subplot(gs[1, :])
    corr_df = df[present].corr(numeric_only=True)
    im = ax_c.imshow(corr_df.values, cmap="RdBu_r", vmin=-1, vmax=1, aspect="auto")
    ax_c.set_xticks(range(len(present)))
    ax_c.set_yticks(range(len(present)))
    ax_c.set_xticklabels(present, rotation=45, ha="right", fontsize=7)
    ax_c.set_yticklabels(present, fontsize=7)
    ax_c.set_title("feature correlation matrix")
    plt.colorbar(im, ax=ax_c, fraction=0.02, pad=0.02)

    path = os.path.join(out_dir, "06_overview.png")
    plt.savefig(path); plt.close()
    print(f"[plot] saved {path}")


# ─── FIGURE 7: per-label time series comparison ───────────────────────────────

def fig_per_label(df, out_dir):
    if "label" not in df.columns:
        return
    print("[plot] figure 7: per-label comparison")
    apply_style()

    metrics = [
        ("ctx_switches",           "ctx switches / interval"),
        ("avg_runq_latency_us",    "runq latency µs"),
        ("avg_syscall_latency_us", "syscall latency µs"),
        ("total_faults",           "page faults / interval"),
        ("total_lock_contentions", "lock contentions / interval"),
        ("hw_llc_misses",          "LLC misses / interval"),
    ]
    metrics = [(m, l) for m, l in metrics if m in df.columns]
    if not metrics:
        return

    n = len(metrics)
    fig, axes = plt.subplots(n, 1, figsize=(14, 3 * n), sharex=False)
    if n == 1: axes = [axes]
    fig.suptitle("Key Metrics by Workload Label", fontsize=13, fontweight="normal", y=1.0)

    labels = sorted(df["label"].unique())
    for ax, (col, ylabel) in zip(axes, metrics):
        for lbl in labels:
            sub = df[df["label"] == lbl].sort_values("t_rel")
            if sub.empty: continue
            agg = sub.groupby("t_rel")[col].sum().reset_index()
            ax.plot(agg["t_rel"], agg[col],
                    color=LABEL_COLORS.get(lbl, MUTED),
                    label=lbl, linewidth=1.2, alpha=0.85)
        ax.set_ylabel(ylabel); ax.grid(True, axis="y"); ax.legend(loc="upper right", fontsize=7)

    plt.tight_layout()
    path = os.path.join(out_dir, "07_per_label.png")
    plt.savefig(path); plt.close()
    print(f"[plot] saved {path}")


# ─── main ─────────────────────────────────────────────────────────────────────

def main():
    parser = argparse.ArgumentParser(description="Plot perf_metrics.csv")
    parser.add_argument("csv", help="path to CSV file")
    parser.add_argument("--pid",   type=int, help="filter to one PID")
    parser.add_argument("--label", help="filter to one label")
    parser.add_argument("--out",   default="./plots", help="output directory for PNGs")
    parser.add_argument("--show",  action="store_true", help="also display windows")
    args = parser.parse_args()

    os.makedirs(args.out, exist_ok=True)

    df = load(args.csv, pid_filter=args.pid, label_filter=args.label)

    fig_scheduling(df, args.out)
    fig_memory(df, args.out)
    fig_syscalls(df, args.out)
    fig_locks(df, args.out)
    fig_hw(df, args.out)
    fig_overview(df, args.out)
    fig_per_label(df, args.out)

    print(f"\n[plot] done. all figures saved to {args.out}/")
    print("[plot] files:")
    for f in sorted(os.listdir(args.out)):
        if f.endswith(".png"):
            print(f"  {args.out}/{f}")

    if args.show:
        matplotlib.use("TkAgg")
        import subprocess
        for f in sorted(os.listdir(args.out)):
            if f.endswith(".png"):
                subprocess.Popen(["xdg-open", os.path.join(args.out, f)])


if __name__ == "__main__":
    main()

ModuleNotFoundError: No module named 'matplotlib'